# SNA LUISS, Intereem report for a network graph regarding Forrest Gump movie.
---
Within this report there would be answering the necessary questions posed per week. There will also be some additional questions answered for fun and cuirocity. 

Due to the demands of the homework, all the functionality of the graph class will be written in an odd way. Given a question part like [(a), find the average degree of the nodes], we will impliment it in a way where in the given weeks question, we write an explicit function that takes in the graph/necessary parameters and show the code, while at the same time, we would write that function into the graph class in ```my_own_graph_library.py``` and in later weeks might call the method from the graph class stright.

In [1]:
## this is the innit sequence, to load the data from the graph in. I also wrote my own graph processing library.
from own_graph_library import Node
from own_graph_library import Edge
from own_graph_library import Graph

import numpy as np
import networkx as nx
import plotly.graph_objects as go



In [2]:
main_graph = Graph()

In [3]:
# loading in node data:
with open("nodes.csv") as f:
    for line in f:
        if line[0] != "#":
            temp_holder = line.strip().split(",")
            new_node = main_graph.add_node(int(temp_holder[0]),temp_holder[1],int(temp_holder[2]))


In [4]:
# loading in edges data:
with open("edges.csv") as f:
    for line in f:
        if line[0] != "#":
            temp_holder = line.strip().split(",")
            new_edge = main_graph.add_edge(int(temp_holder[0]),int(temp_holder[1]),int(temp_holder[2]),int(temp_holder[3]),float(temp_holder[4]))


In [5]:
main_graph.generate_adj_matrix()

In [6]:
# Sample print of the first node with different out degree and out strength

print(main_graph.nodes[14])

index: 14, label: BOY #3, movie_id 316, out degree 3, out strength: 4.0


---
## Week 1
---

(a,b,c) We picked the graph for Forrest Gump, and drew the graph here using networkX and ploty. We implimented our custom graph library to practise and write some of the math functions to algos from scratch

In [7]:
def draw_graph(main_graph):
    """
    Draws the graph using Plotly and NetworkX.
    I caved, I am too lazy to also write my own python GUI library and a custom spacer for nodes, so I stole networkX's code in this area
    """
    G = main_graph.export_to_networkX()  
    
    # for node in main_graph.nodes:
    #     G.add_node(node.index, label=node.label)

    # for edge in main_graph.edges:
    #     G.add_edge(edge.source, edge.target, weight=edge.weight, label=edge.label)

    pos = nx.spring_layout(G) 

    # Create Plotly graph
    edge_x = []
    edge_y = []
    edge_text = []
    for edge in main_graph.edges:
        x0, y0 = pos[edge.source]
        x1, y1 = pos[edge.target]
        edge_x.append(x0)
        edge_x.append(x1)
        edge_y.append(y0)
        edge_y.append(y1)
        edge_text.append(edge.label)

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        line=dict(width=0.5, color='gray'),
        hoverinfo='text',
        text=edge_text,
        mode='lines'
    )

    node_x = []
    node_y = []
    node_text = []
    for node in main_graph.nodes:
        x, y = pos[node.index]
        node_x.append(x)
        node_y.append(y)
        node_text.append(str(node))

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers",
        hoverinfo="text",
        text=node_text,
        marker=dict(
            showscale=True,
            colorscale="YlGnBu",
            size=10,
            colorbar=dict(thickness=15, title="Node connections", xanchor="left", titleside="right")
        )
    )

    # Create layout
    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        showlegend=False,
                        hovermode='closest',
                        title='Network Graph Visualization',
                        xaxis=dict(showgrid=False, zeroline=False),
                        yaxis=dict(showgrid=False, zeroline=False),
                        plot_bgcolor='white'
                    ))
    fig.show()

In [8]:
draw_graph(main_graph)

(d) Compute the number of nodes,edges, average degree and the density. Comment.


In [9]:
# Trivial computations
print(f"Number of Nodes: {len(main_graph.nodes)}") # no. of nodes
print(f"Number of Edges: {len(main_graph.edges)}") # no. of edges


Number of Nodes: 94
Number of Edges: 271


In [10]:
# Average degree out

# Helper function
def count_non_0(arr):
    count = 0
    for i in arr:
        if i > 0:
            count += 1
    return count

def get_avg_degree(main_graph_adj):
    """
    Takes in a 2D array, the adj. matrix of the graph, and returns the count of non-zeros in a row..      
    """
    avg_output = 0
    total_nodes = len(main_graph_adj)
    for i in range(total_nodes):
        non_zero = count_non_0(main_graph_adj[i])
        avg_output += non_zero/total_nodes
    return avg_output

# to deploy this function to calculate the in and out degree we can follow the proposition:
# Given A as Adj. matrix to directed graph G, A^T (A transpose) must be G' where all the directions of the arrows are reversed
# following this;
#  sum A_i = out degree 
#  sum A^T_i = in degree 

# in degree

avg_out_degree = get_avg_degree(main_graph.adj_matrix)
# print("---")
# Once again I was lazy to write my own unoptimised O(n^2) transpose function
adj_matrix_transpose = np.array(main_graph.adj_matrix)
adj_matrix_transpose = np.transpose(adj_matrix_transpose)
# print(main_graph.adj_matrix)
# print(adj_matrix_transpose)


avg_in_degree = get_avg_degree(adj_matrix_transpose)

print(f"average in degree: {avg_in_degree}")
print(f"average out degree: {avg_out_degree}")


average in degree: 2.882978723404256
average out degree: 2.882978723404253


In [11]:
# Density
def density_calc(main_graph):
    edges = len(main_graph.edges)
    nodes = len(main_graph.nodes)
    density = edges/(nodes*(nodes-1))
    return density

print(f"Density: {density_calc(main_graph)}")

Density: 0.030999771219400594


#### Comments:
 - Almost the same avg. in and out degree
 - quite low density, even though edges is almost 2x the density
 - avg in degree and out degree are both close to ~2.8, despite forrest gump being the main character (in which we expect a very hgih connections), indicating huge number of supplemantry cast/charaters 

---
## Week 2
---
While considering the largest component of your network.

(a) Design a function computing the clusering at every nodes and another on that computes the average clustering.

(b) Compare to inbuild function Compute Average clustering and Transitivity number


In [12]:
# for part (a) there are many different ways to define clustering for a DiGraph, so we shall turn it into a undirected, unweighted graph.

# We also have a helper function to check if the main area is "fully connected"

# adj_matrix_new = element_wise_binary(A + A_transpose)

undirected_adj_matrix = main_graph.adj_matrix + np.transpose(main_graph.adj_matrix)
# print(undirected_adj_matrix)

# Naieve implimenatation of simple Clustering checker

def our_clustering(node,adj_matrix):
    # total_nodes = len(node.links)
    edge_found = 0
    all_pariwise_dist = []
    for i in range(len(node.links)):
        # current_node = node.links[i].target
        for j in range(i+1,len(node.links)):
            # print(node.links[i].target,node.links[j].target)
            all_pariwise_dist.append((node.links[i].target,node.links[j].target))
    
    # checking using indexing
    for pair in all_pariwise_dist:
        # since the graph is already 0 indexed, we do not need to subtract 1 from the pair indexes.
        
        # Another thing is we did not turn the graph into a "unweighted graph" in the sense where the i element_of A not strictly in {0,1}
        # This is sloppy programming, but we can skip out a O(n^2) loop by checking if the number is >= 1. Achieves the same as converting all the values to {0,1}
        # but since weight is not use in our clustering it should be mathmatically identical.

        if adj_matrix[pair[0]][pair[1]] >= 1:
            edge_found += 1
    
    # this accounts for the edge case where there the guy has NO FRIENDS.
    if len(all_pariwise_dist) == 0:
        return 0
    else:
        return edge_found/len(all_pariwise_dist)
    
# simple test, changing the index
print(our_clustering(main_graph.nodes[2],undirected_adj_matrix))
print(our_clustering(main_graph.nodes[1],undirected_adj_matrix))
print(our_clustering(main_graph.nodes[0],undirected_adj_matrix))



0.4
0
1.0


In [13]:
# this is the clustering on every node
all_clusterings = []
for node in main_graph.nodes:
    all_clusterings.append(our_clustering(node,undirected_adj_matrix))

print("--- Clustering at every Node ---")
print(all_clusterings)

# avg. clustering

# print("---  at every Node ---")

print(f"average clustering: {sum(all_clusterings)/len(all_clusterings)}")

--- Clustering at every Node ---
[1.0, 0, 0.4, 0.8333333333333334, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7333333333333333, 1.0, 1.0, 0, 1.0, 0.3333333333333333, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.05714285714285714, 1.0, 0, 0, 1.0, 1.0, 0.09666666666666666, 0, 0, 0, 0, 0, 0.3, 0.39285714285714285, 0, 0, 1.0, 1.0, 1.0, 1.0, 1.0, 0, 0, 1.0, 0, 0, 0, 0, 0.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.0, 0, 1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
average clustering: 0.4590070921985816


(b) Comparison with network x

well network x's implimentation is alot smarter! They thought of this as a finding triangles that contain i problem. I will attach their source code here:

```python
@nx._dispatchable(edge_attrs="weight")
def clustering(G, nodes=None, weight=None):
    r"""Compute the clustering coefficient for nodes.

    For unweighted graphs, the clustering of a node :math:`u`
    is the fraction of possible triangles through that node that exist,

    .. math::

      c_u = \frac{2 T(u)}{deg(u)(deg(u)-1)},

    where :math:`T(u)` is the number of triangles through node :math:`u` and
    :math:`deg(u)` is the degree of :math:`u`.

    For weighted graphs, there are several ways to define clustering [1]_.
    the one used here is defined
    as the geometric average of the subgraph edge weights [2]_,

    .. math::

       c_u = \frac{1}{deg(u)(deg(u)-1))}
             \sum_{vw} (\hat{w}_{uv} \hat{w}_{uw} \hat{w}_{vw})^{1/3}.

    The edge weights :math:`\hat{w}_{uv}` are normalized by the maximum weight
    in the network :math:`\hat{w}_{uv} = w_{uv}/\max(w)`.

    The value of :math:`c_u` is assigned to 0 if :math:`deg(u) < 2`.

    Additionally, this weighted definition has been generalized to support negative edge weights [3]_.

    For directed graphs, the clustering is similarly defined as the fraction
    of all possible directed triangles or geometric average of the subgraph
    edge weights for unweighted and weighted directed graph respectively [4]_.

    .. math::

       c_u = \frac{T(u)}{2(deg^{tot}(u)(deg^{tot}(u)-1) - 2deg^{\leftrightarrow}(u))},

    where :math:`T(u)` is the number of directed triangles through node
    :math:`u`, :math:`deg^{tot}(u)` is the sum of in degree and out degree of
    :math:`u` and :math:`deg^{\leftrightarrow}(u)` is the reciprocal degree of
    :math:`u`.


    Parameters
    ----------
    G : graph

    nodes : node, iterable of nodes, or None (default=None)
        If a singleton node, return the number of triangles for that node.
        If an iterable, compute the number of triangles for each of those nodes.
        If `None` (the default) compute the number of triangles for all nodes in `G`.

    weight : string or None, optional (default=None)
       The edge attribute that holds the numerical value used as a weight.
       If None, then each edge has weight 1.

    Returns
    -------
    out : float, or dictionary
       Clustering coefficient at specified nodes

    Examples
    --------
    >>> G = nx.complete_graph(5)
    >>> print(nx.clustering(G, 0))
    1.0
    >>> print(nx.clustering(G))
    {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0}

    Notes
    -----
    Self loops are ignored.

    References
    ----------
    .. [1] Generalizations of the clustering coefficient to weighted
       complex networks by J. Saramäki, M. Kivelä, J.-P. Onnela,
       K. Kaski, and J. Kertész, Physical Review E, 75 027105 (2007).
       http://jponnela.com/web_documents/a9.pdf
    .. [2] Intensity and coherence of motifs in weighted complex
       networks by J. P. Onnela, J. Saramäki, J. Kertész, and K. Kaski,
       Physical Review E, 71(6), 065103 (2005).
    .. [3] Generalization of Clustering Coefficients to Signed Correlation Networks
       by G. Costantini and M. Perugini, PloS one, 9(2), e88669 (2014).
    .. [4] Clustering in complex directed networks by G. Fagiolo,
       Physical Review E, 76(2), 026107 (2007).
    """
    if G.is_directed():
        if weight is not None:
            td_iter = _directed_weighted_triangles_and_degree_iter(G, nodes, weight)
            clusterc = {
                v: 0 if t == 0 else t / ((dt * (dt - 1) - 2 * db) * 2)
                for v, dt, db, t in td_iter
            }
        else:
            td_iter = _directed_triangles_and_degree_iter(G, nodes)
            clusterc = {
                v: 0 if t == 0 else t / ((dt * (dt - 1) - 2 * db) * 2)
                for v, dt, db, t in td_iter
            }
    else:
        # The formula 2*T/(d*(d-1)) from docs is t/(d*(d-1)) here b/c t==2*T
        if weight is not None:
            td_iter = _weighted_triangles_and_degree_iter(G, nodes, weight)
            clusterc = {v: 0 if t == 0 else t / (d * (d - 1)) for v, d, t in td_iter}
        else:
            td_iter = _triangles_and_degree_iter(G, nodes)
            clusterc = {v: 0 if t == 0 else t / (d * (d - 1)) for v, d, t, _ in td_iter}
    if nodes in G:
        # Return the value of the sole entry in the dictionary.
        return clusterc[nodes]
    return clusterc
```
The intuition here is something I could not have came up with, I did not think hard enough about this problem. But if I understand this correctly, their core idea here  to find the clustering for a node *i* we simply ask, out of all the nodes *i* is a part of, how many 3 nodes contain *i*?

so if *i* has nodes neighbours A, B and C

and A has 1 triangle which contain node *i*
and B has 1 triangle which contain node *i*

and there is no other triangle, then it follows the total triangles that can exist is 6(an emergent property of n(n-1)/2), but only 2 exit, clustering is then 1/3.

This is quite smart, since a triangle auto encodes the idea of "a pair of my friends that is my friends".

The transitivity algo in networkX also relies on the same triangle computation.


---
## Week 3
---
(a) Compute the cumulative distribution of the clustering

(b) Define for every node the average of clustering of its neighbors. Compute the cumulative distribution

(c) Compare the two distributions.


In [18]:
# to compute the cumulative distribution of the clusterng

# we sort the list of clusterings computed earlier, and we create a hashmp.

# sorted_all_clusterings = all_clusterings[::] # shorthand deep copy
# sorted_all_clusterings.sort(reverse=1)

# print(sorted_all_clusterings)

# making a hash_map
clustering_count = {}

for i in range(len(all_clusterings)):
    if all_clusterings[i] in clustering_count:
        clustering_count[all_clusterings[i]] += 1
    else:
        clustering_count[all_clusterings[i]] = 1

print(clustering_count)

# sanity check, checking if clustering counts add up to no. of nodes

# print(len(main_graph.nodes))
# sums = 0
# for i in clustering_count:
#     sums += clustering_count[i]
#     print(sums)

# ok I am still sane

# now we sort the clustering count

x = np.array(sorted(clustering_count.keys()))
counts = np.array([clustering_count[k] for k in x])

# Cumulative distribution
cdf = np.cumsum(counts) / counts.sum()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x,
    y=cdf,
    mode="lines",
    line_shape="hv",  # horizontal-vertical step style
    name="CDF"
))

fig.update_layout(
    title="Cumulative Distribution",
    xaxis_title="Value",
    xaxis=dict(range=[max(x), min(x)]),
    yaxis_title="Cumulative Probability",
    yaxis=dict(range=[0, 1]),
    template="plotly_white"
)

fig.show()


{1.0: 40, 0: 46, 0.4: 1, 0.8333333333333334: 1, 0.7333333333333333: 1, 0.3333333333333333: 1, 0.05714285714285714: 1, 0.09666666666666666: 1, 0.3: 1, 0.39285714285714285: 1}


Comments:
There seem to be a bifrucation in the graph, there are then generally 2 groups that we can infer. The first group are the the people with many friends/relations in the movie, (high number of clustering of 1) and there are a large number of people with no friends/relations in the movie.

---
## Week 5
---
1) Depending on what seems more relevant in your graph, pick two of the following local notions

 - Decay centrality
 - Betweeness centrality
 -  Closeness centrality
 - Any other notions that you invent
 -  Pagerank
 
2) Identify the most central nodes.

For the forrest gump movie, well, it is about forrest gump. Hence, by the notion of plot armour (maybe in SNA we can call this the main charater measure), Forrest is the most important person (most central node).

But this is boring, so we can impliment 3 different centrality measures: 
 1. Decay centrality from FORREST node
 2. Betweeness centrality (to see if FORREST node is really the one that ties everyone together)
 3. go invent something

In [ ]:
# Implimenting decay centrality.

# to do so I must first impliment some kind of shortest path finder :/

# Thank you dijkstra

